<a href="https://colab.research.google.com/github/elsa-paul11/de-portfolio-2026/blob/main/module-02-spark-internals/m2_d3_skew_and_salting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install pyspark==4.0.0 -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from decimal import Decimal
import random
import time
import logging
import sys

# Force reconfigure logging
logger = logging.getLogger("day 3")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    ))
    logger.addHandler(handler)

logger.propagate = False  # Prevent Colab's root logger from interfering

logger.info("Logger working correctly")

spark = SparkSession.builder \
    .appName("day3_skew_and_salting") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
logger.info("Spark ready")

2026-06-06 18:36:31,123 | INFO | Logger working correctly
2026-06-06 18:36:31,131 | INFO | Spark ready


In [8]:

# This data is intentionally skewed
# IN-SOUTH has 70% of all rows — simulates real production skew

random.seed(42)

# Weighted regions — IN-SOUTH dominates
regions = ["IN-SOUTH"] * 70 + \
          ["IN-NORTH"] * 10 + \
          ["IN-WEST"]  * 10 + \
          ["IN-EAST"]  * 10

statuses = ["COMPLETED", "PENDING", "CANCELLED", "REFUNDED"]

rows = []
for i in range(1, 100_001):  # 100,000 rows today — more visible skew
    rows.append((
        i,
        random.randint(1, 5000),
        Decimal(str(round(random.uniform(10.0, 5000.0), 2))),
        random.choice(statuses),
        random.choice(regions),
    ))

schema = StructType([
    StructField("order_id",    IntegerType(),     False),
    StructField("customer_id", IntegerType(),     False),
    StructField("amount",      DecimalType(10,2), False),
    StructField("status",      StringType(),      True),
    StructField("region",      StringType(),      True),
])

df_skewed = spark.createDataFrame(rows, schema=schema)

# See the skew clearly
logger.info("Data distribution:")
df_skewed.groupBy("region") \
         .count() \
         .withColumn("percentage",
             F.round(F.col("count") / 100000 * 100, 1)) \
         .orderBy(F.col("count").desc()) \
         .show()

2026-06-06 18:36:37,144 | INFO | Data distribution:
+--------+-----+----------+
|  region|count|percentage|
+--------+-----+----------+
|IN-SOUTH|70280|      70.3|
|IN-NORTH|10070|      10.1|
| IN-WEST| 9933|       9.9|
| IN-EAST| 9717|       9.7|
+--------+-----+----------+



In [9]:
# Normal groupBy — skewed partition does all the work

start = time.time()

result_skewed = df_skewed \
    .groupBy("region") \
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("order_count")
    )

result_skewed.show()
time_skewed = time.time() - start

logger.info(f"Skewed groupBy time: {time_skewed:.3f} seconds")

+--------+------------+-----------+
|  region|total_amount|order_count|
+--------+------------+-----------+
| IN-WEST| 25006454.81|       9933|
| IN-EAST| 24240420.33|       9717|
|IN-NORTH| 25145901.44|      10070|
|IN-SOUTH|175337072.44|      70280|
+--------+------------+-----------+

2026-06-06 18:36:44,638 | INFO | Skewed groupBy time: 2.039 seconds


In [10]:
# SALTING — spread skewed key across multiple partitions
SALT_COUNT = 4  # number of buckets to split skewed data into

start = time.time()

# Step 1: Add random salt to each row
df_salted = df_skewed.withColumn(
    "salt",
    (F.rand() * SALT_COUNT).cast(IntegerType())
).withColumn(
    "salted_region",
    F.concat(F.col("region"), F.lit("_"), F.col("salt").cast(StringType()))
)

# Step 2: Partial aggregation using salted key
# IN-SOUTH_0, IN-SOUTH_1, IN-SOUTH_2, IN-SOUTH_3 are now separate keys
partial_agg = df_salted \
    .groupBy("salted_region", "region") \
    .agg(
        F.sum("amount").alias("partial_sum"),
        F.count("*").alias("partial_count")
    )

# Step 3: Remove salt, final aggregation
result_salted = partial_agg \
    .groupBy("region") \
    .agg(
        F.sum("partial_sum").alias("total_amount"),
        F.sum("partial_count").alias("order_count")
    )

result_salted.show()
time_salted = time.time() - start

logger.info(f"Salted groupBy time:  {time_salted:.3f} seconds")
logger.info(f"Skewed groupBy time:  {time_skewed:.3f} seconds")
logger.info(f"Improvement:          {((time_skewed - time_salted)/time_skewed*100):.1f}% faster")

+--------+------------+-----------+
|  region|total_amount|order_count|
+--------+------------+-----------+
| IN-WEST| 25006454.81|       9933|
| IN-EAST| 24240420.33|       9717|
|IN-NORTH| 25145901.44|      10070|
|IN-SOUTH|175337072.44|      70280|
+--------+------------+-----------+

2026-06-06 18:36:49,921 | INFO | Salted groupBy time:  1.537 seconds
2026-06-06 18:36:49,927 | INFO | Skewed groupBy time:  2.039 seconds
2026-06-06 18:36:49,932 | INFO | Improvement:          24.6% faster


In [11]:
# See how salting redistributes data across partitions

print("=== WITHOUT SALTING ===")
df_skewed.groupBy("region") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show()

print("=== WITH SALTING (salted keys) ===")
df_salted.groupBy("salted_region") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show()

=== WITHOUT SALTING ===
+--------+-----+
|  region|count|
+--------+-----+
|IN-SOUTH|70280|
|IN-NORTH|10070|
| IN-WEST| 9933|
| IN-EAST| 9717|
+--------+-----+

=== WITH SALTING (salted keys) ===
+-------------+-----+
|salted_region|count|
+-------------+-----+
|   IN-SOUTH_2|17649|
|   IN-SOUTH_1|17594|
|   IN-SOUTH_0|17565|
|   IN-SOUTH_3|17472|
|   IN-NORTH_0| 2596|
|    IN-WEST_2| 2594|
|   IN-NORTH_3| 2536|
|    IN-WEST_3| 2517|
|   IN-NORTH_2| 2500|
|    IN-EAST_3| 2482|
|    IN-EAST_0| 2441|
|   IN-NORTH_1| 2438|
|    IN-WEST_0| 2413|
|    IN-WEST_1| 2409|
|    IN-EAST_1| 2406|
|    IN-EAST_2| 2388|
+-------------+-----+

